# Finetuning ESMC with LoRA

In this notebook, we will learn how to finetune ESMC to classify enzymes by EC number using LoRA (Low-rank adaptation) for parameter efficient finetuning. 
For an overview of LoRA, see the [PEFT LoRA documentation](https://huggingface.co/docs/peft/package_reference/lora).

## Imports

In [ ]:
# !pip install peft

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

In [ ]:
from transformers import AutoTokenizer, ESMCForSequenceClassification

## Download the dataset

In [ ]:
# download the CARE enzyme classification dataset from [Yang et al, 2024](https://proceedings.neurips.cc/paper_files/paper/2024/hash/05a7ad45d75a3082d7a3a70de8743140-Abstract-Datasets_and_Benchmarks_Track.html)

# !wget https://zenodo.org/records/14004425/files/CARE_datasets.zip
# !unzip CARE_datasets.zip

In [ ]:
PROTEIN_TRAIN_CSV = "./CARE_datasets/splits/task1/protein_train.csv"
PROTEIN_TEST_CSV = "./CARE_datasets/splits/task1/30-50_protein_test.csv"

There are several test datasets, see the [CARE github](https://github.com/jsunn-y/CARE/tree/main) for more information. The one we have chosen here has been filtered to be 30-50% sequence identical to the training set.

In [ ]:
def load_protein_train():
    df_train = pd.read_csv(PROTEIN_TRAIN_CSV)
    return df_train


def load_protein_test():
    df_test = pd.read_csv(PROTEIN_TEST_CSV)
    return df_test


def get_stratified_train_and_val(rng, val_fraction=0.1):
    """Load the CARE train CSV and split it into stratified train / val sets.

    Each EC1 class's indices are shuffled independently and ``val_fraction`` of
    them are held out, so the validation set preserves the train class mix.

    Parameters
    ----------
    rng : np.random.Generator
        Random generator used to shuffle each class's indices.
    val_fraction : float, optional
        Fraction of each class to assign to validation.

    Returns
    -------
    train_sequences, train_labels, val_sequences, val_labels
    """
    train_df = load_protein_train()
    ec1_classes = sorted(train_df["EC1"].unique())
    ec1_to_idx = {ec: i for i, ec in enumerate(ec1_classes)}

    all_sequences = train_df["Sequence"].tolist()
    all_labels = np.array([ec1_to_idx[ec] for ec in train_df["EC1"].to_numpy()])

    val_mask = np.zeros(len(all_labels), dtype=bool)
    for idx in ec1_to_idx.values():
        class_positions = np.where(all_labels == idx)[0]
        rng.shuffle(class_positions)
        n_val = int(round(len(class_positions) * val_fraction))
        val_mask[class_positions[:n_val]] = True

    train_sequences = [s for s, m in zip(all_sequences, val_mask) if not m]
    train_labels = all_labels[~val_mask]
    val_sequences = [s for s, m in zip(all_sequences, val_mask) if m]
    val_labels = all_labels[val_mask]

    return train_sequences, train_labels, val_sequences, val_labels


def plot_train_test_ec_distribution(
    train_df: pd.DataFrame, test_df: pd.DataFrame, ec1_classes: list[int]
) -> None:
    print(f"Train: {len(train_df):,} proteins")
    print(f"Test:  {len(test_df):,} proteins")

    train_counts = train_df["EC1"].value_counts().reindex(ec1_classes, fill_value=0)
    test_counts = test_df["EC1"].value_counts().reindex(ec1_classes, fill_value=0)

    x = np.arange(len(ec1_classes))
    xtick_labels = [f"{c}. {EC1_NAMES[c]}" for c in ec1_classes]

    fig, (ax_train, ax_test) = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    ax_train.bar(x, train_counts.values, color="C0")
    ax_train.set_title(f"train (n={len(train_df):,})")
    ax_train.set_ylabel("number of proteins")
    ax_test.bar(x, test_counts.values, color="C1")
    ax_test.set_title(f"test (n={len(test_df):,})")

    for ax in (ax_train, ax_test):
        ax.set_xticks(x)
        ax.set_xticklabels(xtick_labels, rotation=30, ha="right")

    fig.suptitle("CARE Task 1: EC1 label distribution")
    fig.tight_layout()
    plt.show()

## Visualize the dataset

For simplicity, we are using the top-level Enzyme Commission class (EC1), giving 7 classes:

1. Oxidoreductases
2. Transferases
3. Hydrolases
4. Lyases
5. Isomerases
6. Ligases
7. Translocases

In [ ]:
EC1_NAMES = {
    1: "Oxidoreductases",
    2: "Transferases",
    3: "Hydrolases",
    4: "Lyases",
    5: "Isomerases",
    6: "Ligases",
    7: "Translocases",
}

In [ ]:
train_df = load_protein_train()
test_df = load_protein_test()
ec1_classes = sorted(EC1_NAMES)

plot_train_test_ec_distribution(train_df, test_df, ec1_classes)

# Stratified train / val split

We want a held-out validation set to track generalization during training. We use stratified sampling by EC number so the val set preserves the class mix of the train set.

In [ ]:
VAL_FRACTION = 0.01
SPLIT_SEED = 0

In [ ]:
ec1_to_idx = {ec: i for i, ec in enumerate(ec1_classes)}
test_sequences = test_df["Sequence"].tolist()
test_labels = np.array([ec1_to_idx[ec] for ec in test_df["EC1"].to_numpy()])

split_rng = np.random.default_rng(SPLIT_SEED)
train_sequences, train_labels, val_sequences, val_labels = get_stratified_train_and_val(
    split_rng, val_fraction=VAL_FRACTION
)

print(
    f"Train: {len(train_sequences):,}  Val: {len(val_sequences):,}  Test: {len(test_sequences):,}"
)
print(
    "Val EC1 distribution:",
    {ec: int((val_labels == i).sum()) for ec, i in ec1_to_idx.items()},
)

# Load tokenizer and model with LoRA

Load the pretrained ESMC backbone with a fresh sequence-classification head, then wrap with LoRA adapters. Only the LoRA parameters and the new classifier head are trained — the backbone weights stay frozen.

## The sequence classification model

`ESMCForSequenceClassification` wraps the pretrained ESMC encoder with a small classification head:

1. **ESMC backbone** — produces a per-token hidden state of shape `(batch, seq_len, d_model)`.
2. **`<cls>` pooling** — the head takes the hidden state at position 0 (the `<cls>` token prepended by the tokenizer) as the sequence-level representation.
3. **Classification head** — `Linear(d_model, d_model)` → `tanh` → `Dropout` → `Linear(d_model, num_labels)`, producing one logit per class.

When we call `.from_pretrained(MODEL_PATH, num_labels=7)`, the backbone weights are loaded from the ESMC checkpoint and the two `Linear` layers in the head are freshly initialized — you'll see a HuggingFace warning to that effect, which is expected. The head's weights are what we'll train (along with the LoRA adapters injected into the backbone in the next step).

For regression, pass `num_labels=1` and float labels.

In [ ]:
MODEL_PATH = "biohub/ESMC-300M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = ESMCForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=len(ec1_classes), device_map="auto"
)

In [ ]:
LORA_RANK = 8
LORA_ALPHA = 16

# layernorm_qkv and ffn fused weights are bare nn.Parameter tensors (not Linear
# submodules), so they are reached via target_parameters rather than target_modules.
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.01,
    target_modules=["out_proj"],
    target_parameters=["layernorm_qkv.weight", "ffn.fc1_weight", "ffn.fc2_weight"],
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_config)
print("\n=== Trainable Parameters ===")
model.print_trainable_parameters()

device = next(model.parameters()).device

# Helpers for training and evaluation

In [ ]:
MAX_SEQ_LEN = 1024
EVAL_BATCH_SIZE = 16

In [ ]:
def make_batch(sequences: list[str], labels: np.ndarray) -> dict[str, torch.Tensor]:
    inputs = tokenizer(
        sequences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    return {
        "input_ids": inputs["input_ids"].to(device),
        "attention_mask": inputs["attention_mask"].to(device),
        "labels": torch.tensor(labels, dtype=torch.long, device=device),
    }


def evaluate(
    model: torch.nn.Module,
    name: str,
    sequences: list[str],
    labels: np.ndarray,
    batch_size: int = EVAL_BATCH_SIZE,
) -> tuple[np.ndarray, float, float]:
    all_preds = np.empty(len(sequences), dtype=np.int64)
    all_losses: list[float] = []
    model.eval()
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for start in range(0, len(sequences), batch_size):
            end = start + batch_size
            batch = make_batch(sequences[start:end], labels[start:end])
            outputs = model(**batch)
            all_losses.append(outputs.loss.item() * (end - start))
            all_preds[start:end] = outputs.logits.argmax(dim=-1).cpu().numpy()
    loss = float(np.sum(all_losses) / len(sequences))
    acc = float((all_preds == labels).mean())
    print(f"{name}: loss={loss:.4f} acc={acc:.4f}")
    return all_preds, loss, acc


def plot_training_metrics(
    losses: list[float],
    accuracies: list[float],
    val_history: list[dict],
    window: int = 20,
) -> None:
    steps = np.arange(1, len(losses) + 1)
    have_smooth = len(losses) >= window
    if have_smooth:
        loss_smooth = np.convolve(losses, np.ones(window) / window, mode="valid")
        acc_smooth = np.convolve(accuracies, np.ones(window) / window, mode="valid")
        smooth_steps = steps[window - 1 :]

    val_steps = np.array([h["step"] for h in val_history])
    val_losses = np.array([h["loss"] for h in val_history])
    val_accs = np.array([h["acc"] for h in val_history])

    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))

    ax_loss.plot(steps, losses, color="C0", alpha=0.3, label="train (per-step)")
    if have_smooth:
        ax_loss.plot(
            smooth_steps, loss_smooth, color="C0", label=f"train (rolling w={window})"
        )
    if len(val_steps) > 0:
        ax_loss.plot(val_steps, val_losses, color="C2", marker="o", label="val")
    ax_loss.set_xlabel("step")
    ax_loss.set_ylabel("loss")
    ax_loss.set_title("loss")
    ax_loss.legend()

    ax_acc.plot(steps, accuracies, color="C1", alpha=0.3, label="train (per-step)")
    if have_smooth:
        ax_acc.plot(
            smooth_steps, acc_smooth, color="C1", label=f"train (rolling w={window})"
        )
    if len(val_steps) > 0:
        ax_acc.plot(val_steps, val_accs, color="C2", marker="o", label="val")
    ax_acc.set_xlabel("step")
    ax_acc.set_ylabel("accuracy")
    ax_acc.set_ylim(0, 1)
    ax_acc.set_title("accuracy")
    ax_acc.legend()

    fig.tight_layout()
    plt.show()


def plot_confusion_matrix(
    labels: np.ndarray, preds: np.ndarray, ec1_classes: list[int], test_acc: float
) -> None:
    confusion = np.zeros((len(ec1_classes), len(ec1_classes)), dtype=int)
    for true, pred in zip(labels, preds):
        confusion[true, pred] += 1

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(confusion, cmap="Blues")
    ax.set_xticks(range(len(ec1_classes)))
    ax.set_yticks(range(len(ec1_classes)))
    ax.set_xticklabels(
        [f"{c}. {EC1_NAMES[c]}" for c in ec1_classes], rotation=30, ha="right"
    )
    ax.set_yticklabels([f"{c}. {EC1_NAMES[c]}" for c in ec1_classes])
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(f"EC1 confusion matrix (test acc={test_acc:.3f})")

    vmax = confusion.max()
    for i in range(len(ec1_classes)):
        for j in range(len(ec1_classes)):
            ax.text(
                j,
                i,
                int(confusion[i, j]),
                ha="center",
                va="center",
                color="white" if confusion[i, j] > vmax / 2 else "black",
                fontsize=8,
            )
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    plt.show()

# Training Loop

In [ ]:
# Adjust NUM_TRAINING_STEPS to change how long training runs. We recommend ~5000 steps
# for a meaningful finetune; 1000 is used here so the notebook completes quickly.
# 1000 steps takes about 4 minutes on an H100.
NUM_TRAINING_STEPS = 1000
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
SEED = 0
VAL_EVERY_STEPS = 250

In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)

rng = np.random.default_rng(SEED)
n_train = len(train_sequences)

losses: list[float] = []
accuracies: list[float] = []
val_history: list[dict] = []
postfix = {"train_loss": "n/a", "train_acc": "n/a", "val_loss": "n/a", "val_acc": "n/a"}
model.train()

perm = rng.permutation(n_train)
pbar = tqdm(range(NUM_TRAINING_STEPS), desc="training")
for step in pbar:
    start = (step * BATCH_SIZE) % n_train
    if start + BATCH_SIZE > n_train:
        perm = rng.permutation(n_train)
        start = 0
    idx = perm[start : start + BATCH_SIZE]
    batch = make_batch([train_sequences[int(i)] for i in idx], train_labels[idx])

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        outputs = model(**batch)
        # our default model is intialized with a CrossEntropy loss.
        # to experiment with the loss, you can manually compute it here:
        # loss = my_loss-fn(outputs.logits, batch["labels"])
        loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    preds = outputs.logits.argmax(dim=-1)
    acc = (preds == batch["labels"]).float().mean().item()
    losses.append(loss.item())
    accuracies.append(acc)

    if (step + 1) % 20 == 0:
        postfix["train_loss"] = f"{np.mean(losses[-20:]):.3f}"
        postfix["train_acc"] = f"{np.mean(accuracies[-20:]):.2f}"
        pbar.set_postfix(postfix)

    if (step + 1) % VAL_EVERY_STEPS == 0 and len(val_sequences) > 0:
        _, val_loss, val_acc = evaluate(model, "val", val_sequences, val_labels)
        val_history.append({"step": step + 1, "loss": val_loss, "acc": val_acc})
        postfix["val_loss"] = f"{val_loss:.3f}"
        postfix["val_acc"] = f"{val_acc:.2f}"
        pbar.set_postfix(postfix)
        model.train()

In [ ]:
plot_training_metrics(losses, accuracies, val_history)

# Evaluation on test data

Evaluate the LoRA-fine-tuned ESMC prediction model on the held-out CARE test split and plot the EC1 confusion matrix. Rows are true classes, columns are predicted classes — diagonal cells are correct predictions, off-diagonal cells show where the model confuses classes.

In [ ]:
preds, test_loss, test_acc = evaluate(model, "test", test_sequences, test_labels)

plot_confusion_matrix(test_labels, preds, ec1_classes, test_acc)

In 1000 steps (a full epoch would be ~6K) we can already classify the most dominant classes pretty well, but some are being missed entirely. If you have time, you can go back and train for a full epoch. Now you can take this code and extend it to your use-case, for example:

* experimenting with contrastive learning objectives (e.g. [CLEAN, Yu, et al. 2023](https://www.science.org/doi/10.1126/science.adf2465))
* learning a regression head to predict the outcome of experimental data
* continuing masked language modeling on antibody sequences from [OAS](https://opig.stats.ox.ac.uk/webapps/oas/) to learn a specialist language model

Some tasks may require careful tuning of the LoRA rank and alpha parameters, as well as learning rates. Make sure that you experiment for your particular task to understand how much capacity you want in your finetuned model.
